# Group Classifier Inference

Run the inference pipeline and write prediction parquet output to `data/group_classifier`.

In [1]:
from pathlib import Path

from pioneerml.common.zenml import load_step_output
from pioneerml.common.zenml import utils as zenml_utils
from pioneerml.pipelines.inference.group_classification import group_classification_inference_pipeline

PROJECT_ROOT = zenml_utils.find_project_root()
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


In [2]:
# Inputs
data_dir = Path(PROJECT_ROOT) / "data"
parquet_paths = sorted(data_dir.glob("ml_output_*.parquet"))
parquet_paths = parquet_paths[:1]  # optional subset
parquet_paths = [str(p.resolve()) for p in parquet_paths]
if not parquet_paths:
    raise RuntimeError(...)

model_path = None # Uses latest model path. Example to use custom model path: str((PROJECT_ROOT / 'trained_models' / 'groupclassifier' / 'groupclassifier_20260208_223249_torchscript.pt').resolve())
output_dir = str((PROJECT_ROOT / 'data' / 'group_classifier').resolve())


In [3]:
# Run inference pipeline
# Ensure JSON-serializable pipeline inputs
parquet_paths = [str(p) for p in parquet_paths]

run = group_classification_inference_pipeline.with_options(enable_cache=False)(
    parquet_paths=parquet_paths,
    model_path=model_path,
    output_dir=output_dir,
    pipeline_config={
        'loader': {'config_json': {'batch_size': 64, 'chunk_row_groups': 4, 'chunk_workers': 0}},
        'inference': {'threshold': 0.5},
        'save_predictions': {'check_accuracy': False, 'write_timestamped': False},
    },
)



Initiating a new run for the pipeline: group_classification_inference_pipeline.
Caching is disabled by default for group_classification_inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step load_group_classifier_inference_inputs has started.
Step load_group_classifier_inference_inputs has finished in 0.207s.
Step load_group_classifier_model has started.
Step load_group_classifier_model has finished in 0.113s.
Step run_group_classifier_inference has started.
Step run_group_classifier_inference has finished in 5.126s.
Step save_group_classifier_predictions has started.
Step save_group_classifier_predictions has finished in 0.372s.
Pipeline run has finished in 8.473s.


In [4]:
# Inspect export outputs
export = load_step_output(run, "save_group_classifier_predictions")
print("export:", export)

predictions_paths = [Path(p) for p in (export.get("predictions_paths") or [])]
if not predictions_paths and export.get("predictions_path"):
    predictions_paths = [Path(export["predictions_path"])]

print("predictions_paths:")
for p in predictions_paths:
    print(" ", p)


export: {'predictions_path': '/workspace/data/group_classifier/ml_output_000_preds.parquet', 'predictions_paths': ['/workspace/data/group_classifier/ml_output_000_preds.parquet'], 'timestamped_predictions_path': None, 'timestamped_predictions_paths': [], 'num_rows': 1024}
predictions_paths:
  /workspace/data/group_classifier/ml_output_000_preds.parquet


In [5]:
# Optional: verify parquet schema + small sample (avoids loading full file)
import gc
import pyarrow as pa
import pyarrow.parquet as pq

if not predictions_paths:
    raise RuntimeError("No prediction parquet files were exported.")

pf = pq.ParquetFile(predictions_paths[0])
print("file:", predictions_paths[0])
print("rows:", pf.metadata.num_rows)
print(pf.schema_arrow)

if pf.num_row_groups > 0:
    sample = pf.read_row_group(0).slice(0, 3)
    print(sample)
else:
    sample = None
    print("No row groups found.")

# Release notebook-held references after inspection
del sample, pf
gc.collect()
pa.default_memory_pool().release_unused()


file: /workspace/data/group_classifier/ml_output_000_preds.parquet
rows: 1024
pred_pion: list<element: float>
  child 0, element: float
pred_muon: list<element: float>
  child 0, element: float
pred_mip: list<element: float>
  child 0, element: float
pyarrow.Table
pred_pion: list<element: float>
  child 0, element: float
pred_muon: list<element: float>
  child 0, element: float
pred_mip: list<element: float>
  child 0, element: float
----
pred_pion: [[[0.99945146,0.00044046453,0.00020653607],[0.9967438,0.000035299956],[0.99996316,9.890877e-7,0.999956,0.03963916,0.0015000806]]]
pred_muon: [[[0.029697787,0.99979824,0.0003201145],[0.4319141,0.9999939],[0.02297402,0.9999989,0.012119045,0.94418395,0.00042328794]]]
pred_mip: [[[0.00024944838,0.00008478662,0.9993761],[0.00018079608,5.534124e-7],[0.000008513442,6.912861e-8,0.000012034367,0.015671069,0.99818367]]]
